<a href="https://colab.research.google.com/github/belokonr/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/lab13/lab13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/belokonr/ECON5200-Applied-Data-Analytics-in-Economics/refs/heads/main/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        18:38:12   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [12]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# =============================================================================
# 1. GENERATE SYNTHETIC DATA
#    Simulating a realistic housing dataset with two predictors.
# =============================================================================
np.random.seed(42)
n = 200

property_age = np.random.uniform(1, 60, n)          # years
distance_to_tech_hub = np.random.uniform(0.5, 30, n) # miles

# True relationship: Sale_Price decreases with age and distance, plus noise
sale_price = (
    450_000
    - 1_800 * property_age
    - 4_500 * distance_to_tech_hub
    + np.random.normal(0, 25_000, n)
)

df = pd.DataFrame({
    "Property_Age": property_age,
    "Distance_to_Tech_Hub": distance_to_tech_hub,
    "Sale_Price": sale_price,
})

# =============================================================================
# 2. FIT THE MULTIVARIATE OLS MODEL
#    Sale_Price = β0 + β1·Property_Age + β2·Distance_to_Tech_Hub + ε
# =============================================================================
X = df[["Property_Age", "Distance_to_Tech_Hub"]]
X = sm.add_constant(X)  # adds the intercept column (β0)
y = df["Sale_Price"]

model = sm.OLS(y, X).fit()
print(model.summary())

# -----------------------------------------------------------------------------
# Extract the fitted coefficients from the results object.
#   params[0] → β0 (intercept)
#   params[1] → β1 (coefficient for Property_Age)
#   params[2] → β2 (coefficient for Distance_to_Tech_Hub)
# -----------------------------------------------------------------------------
b0 = model.params.iloc[0]  # intercept
b1 = model.params.iloc[1]  # slope for Property_Age
b2 = model.params.iloc[2]  # slope for Distance_to_Tech_Hub

print(f"\nExtracted coefficients:")
print(f"  Intercept (β0):            {b0:>12,.2f}")
print(f"  Property_Age (β1):         {b1:>12,.2f}")
print(f"  Distance_to_Tech_Hub (β2): {b2:>12,.2f}")

# =============================================================================
# 3. BUILD THE MESHGRID FOR THE REGRESSION SURFACE
#
#    A meshgrid creates every combination of (Property_Age, Distance_to_Tech_Hub)
#    across evenly spaced values spanning each variable's observed range.
#
#    np.linspace creates 30 evenly spaced points for each axis.
#    np.meshgrid then produces two 30×30 matrices:
#      - age_grid[i, j]  holds the Property_Age   value at grid cell (i, j)
#      - dist_grid[i, j] holds the Distance        value at grid cell (i, j)
#
#    We then compute the predicted Sale_Price at every grid cell using:
#      ŷ = β0 + β1·age_grid + β2·dist_grid
#    This gives us a 30×30 matrix of predicted values — our 2D hyperplane.
# =============================================================================
resolution = 30  # number of points per axis (30×30 = 900 surface points)

age_range = np.linspace(
    df["Property_Age"].min(),
    df["Property_Age"].max(),
    resolution,
)
dist_range = np.linspace(
    df["Distance_to_Tech_Hub"].min(),
    df["Distance_to_Tech_Hub"].max(),
    resolution,
)

# Create the two 2-D coordinate matrices from the 1-D axis vectors
age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Predicted surface: apply the linear equation element-wise across the grid
price_surface = b0 + b1 * age_grid + b2 * dist_grid

# =============================================================================
# 4. COMPUTE RESIDUALS FOR COLOR-CODING THE SCATTER POINTS
#    Residual = Actual − Predicted.  Positive → underpriced by model,
#    Negative → overpriced by model.
# =============================================================================
df["Predicted"] = model.fittedvalues
df["Residual"] = model.resid

# =============================================================================
# 5. BUILD THE INTERACTIVE 3-D PLOTLY FIGURE
# =============================================================================

# --- 5a. Scatter: actual data points, colored by residual magnitude ----------
scatter = go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    name="Observed Data",
    marker=dict(
        size=4,
        color=df["Residual"],            # color encodes the residual
        colorscale="RdBu",               # red = negative, blue = positive
        cmid=0,                           # center the colorscale at 0
        colorbar=dict(
            title="Residual ($)",
            thickness=15,
            x=1.02,
        ),
        opacity=0.8,
        line=dict(width=0.3, color="black"),
    ),
    # Hover text shows all key values for each observation
    hovertemplate=(
        "<b>Property Age:</b> %{x:.1f} yrs<br>"
        "<b>Distance:</b> %{y:.1f} mi<br>"
        "<b>Sale Price:</b> $%{z:,.0f}<br>"
        "<b>Residual:</b> $%{text}<br>"
        "<extra></extra>"
    ),
    text=[f"{r:,.0f}" for r in df["Residual"]],
)

# --- 5b. Surface: the fitted regression hyperplane ---------------------------
surface = go.Surface(
    x=age_grid,        # 30×30 matrix of Property_Age values
    y=dist_grid,       # 30×30 matrix of Distance values
    z=price_surface,   # 30×30 matrix of predicted Sale_Price values
    name="OLS Hyperplane",
    colorscale=[[0, "rgba(255, 165, 0, 0.45)"], [1, "rgba(255, 69, 0, 0.45)"]],
    showscale=False,
    opacity=0.55,
    hovertemplate=(
        "<b>Age:</b> %{x:.1f} yrs<br>"
        "<b>Distance:</b> %{y:.1f} mi<br>"
        "<b>Predicted Price:</b> $%{z:,.0f}<br>"
        "<extra>Regression Plane</extra>"
    ),
)

# --- 5c. Assemble the figure with a polished layout -------------------------
fig = go.Figure(data=[surface, scatter])

fig.update_layout(
    title=dict(
        text=(
            "Multivariate OLS · Sale Price ~ Property Age + Distance to Tech Hub"
            f"<br><sup>ŷ = {b0:,.0f}  {b1:,.0f}·Age  {b2:,.0f}·Distance  |  "
            f"R² = {model.rsquared:.3f}</sup>"
        ),
        x=0.5,
        font=dict(size=16),
    ),
    scene=dict(
        xaxis_title="Property Age (years)",
        yaxis_title="Distance to Tech Hub (miles)",
        zaxis_title="Sale Price ($)",
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9)),  # good default angle
        aspectratio=dict(x=1, y=1, z=0.8),
    ),
    legend=dict(
        x=0.01, y=0.99,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="gray",
        borderwidth=1,
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    width=950,
    height=700,
)

fig.show()

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.806
Model:                            OLS   Adj. R-squared:                  0.804
Method:                 Least Squares   F-statistic:                     409.2
Date:                Fri, 13 Mar 2026   Prob (F-statistic):           7.12e-71
Time:                        18:40:37   Log-Likelihood:                -2305.9
No. Observations:                 200   AIC:                             4618.
Df Residuals:                     197   BIC:                             4628.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 4.521e+05 